# 02 · Variables para el recomendador

**Entrada:** `data/processed/recetas_limpias.csv` (salida del notebook 01).
**Salida:** `data/processed/recetas_modelo.csv`, una fila por receta con las columnas que usa el recomendador.

Qué se calcula (detalle de cada decisión en los comentarios del código):

* **Ingredientes base** (`ingredients_core`, `n_ingredients`): nombre del ingrediente sin cantidad, unidad ni corte; singular decidido con los datos; "sal y pimienta" se parte en dos.
* **Esfuerzo y dificultad** (`n_steps`, `effort_score`, `difficulty_level`).
* **Tiempo** (`total_time_min` real; `time_filled_min` estimado con recetas parecidas; `is_quick`).
* **Etiquetas** una columna 0/1 por etiqueta (`tag_*`), país agrupado y banderas del texto (`is_spicy`, `needs_oven`...).
* **`recipe_text`**: el único texto que lee el modelo TF-IDF (nombre + ingredientes base + etiquetas + país).
* **`recipe_key`**: identificador estable por receta.

`rating` se descarta a propósito: solo ~35 % de las recetas lo trae y casi no se relaciona con nada que use el modelo.

In [ ]:
import hashlib
import re
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data" / "processed").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))                       # para poder importar src/

INPUT_PATH = ROOT / "data" / "processed" / "recetas_limpias.csv"
OUTPUT_PATH = ROOT / "data" / "processed" / "recetas_modelo.csv"

# Misma forma de leer el texto que usa la app (minúsculas sin acentos, y separar pasos).
from src.text_utils import normalize_text, split_steps

df = pd.read_csv(INPUT_PATH)                        # cada fila es UNA receta
print(f"Recetas: {len(df):,}")

## 1. Ingredientes base y pasos

In [ ]:
TAGS = ["Desayuno", "Comida", "Cena", "Postre", "Bebida", "Alcohol", "Vegetariana", "Cerdo",
        "Pollo", "Mariscos", "Res", "Vegana", "Pavo", "Cordero", "Pasta", "Sopa"]     # las 16 etiquetas de la app
MEAL_TAGS = ["Desayuno", "Comida", "Cena", "Postre", "Bebida"]
PROTEIN_TAGS = ["Cerdo", "Pollo", "Mariscos", "Res", "Pavo", "Cordero"]

DIFFICULTY_LEVEL = {"facil": 1, "intermedio": 2, "reto": 3}
eda = df.copy()
eda["difficulty_level"] = eda["difficulty"].map(DIFFICULTY_LEVEL)

In [ ]:
# Coma que SEPARA ingredientes en Yanuq: una coma pegada al texto siguiente ("...,2 tomates") y que no sea un decimal ("1,5 kg").
YANUQ_SEPARATOR = re.compile(r"(?<!\d),(?!\s)|(?<=\d),(?!\s|\d)")
# Coma que separa en El Mueble, Vitónica y Directo al Paladar: coma + espacio (y no un decimal).
COMMA_SEPARATOR = re.compile(r"(?<!\d),|,(?!\d)")


def split_ingredient_lines(raw, source):
    """Convierte el texto de `ingredients` en una lista de renglones, usando el separador que cada fuente usa."""
    if not isinstance(raw, str) or not raw.strip():
        return []
    if "|" in raw:                                   # RecetasGratis, Kiwilimon y Cookpad separan con " | "
        parts = raw.split("|")
    elif source == "Yanuq":                          # coma pegada al siguiente ingrediente
        parts = YANUQ_SEPARATOR.split(raw)
    else:                                            # El Mueble, Vitónica, Directo al Paladar: ", "
        parts = COMMA_SEPARATOR.split(raw)
    return [p.strip() for p in parts if p.strip()]


# ---- Renglones que NO son un ingrediente -------------------------------------------
# Varias recetas meten en la misma lista los títulos de sus secciones ("Para la salsa")
# y el rendimiento ("Para 4 personas"). No se compran ni se cocinan: si los dejamos,
# el modelo los contaría como comida y "para la salsa" se volvería un "ingrediente".
SECTION_HEADER = re.compile(r"""(?ix)
    ^\s*(?:
        para\s+\d+\s*(?:[-–aoy/]\s*\d+)?\s*(?:personas?|comensales|raciones?|porciones?)  # "Para 4 personas"
      | para\s+(?:el|la|los|las)\b[^|]{0,30}                                                # "Para la masa"
      | (?:ingredientes?|relleno|decoraci[oó]n|montaje|cobertura|glaseado|guarnici[oó]n|
         acompa[ñn]amiento|elaboraci[oó]n|preparaci[oó]n|otros?|adem[aá]s|opcional(?:es)?)   # títulos sueltos
    )\s*:?\s*$
""")


def is_ingredient_line(line):
    """False si el renglón es un título de sección o una nota de porciones."""
    line = line.strip()
    if line.endswith(":"):                 # "Para los frijoles:" siempre es un título
        return False
    return SECTION_HEADER.match(line) is None

In [ ]:
# Unidades de medida y envases que aparecen entre la cantidad y el ingrediente ("2 TAZAS de harina").
UNITS = """gramos gramo gr grs g kilogramos kilogramo kilos kilo kg miligramos mililitros mililitro ml litros litro l cc
centimetros cubicos onzas onza oz libras libra tazas taza vasos vaso copas copa cucharadas cucharada cucharaditas
cucharadita cuchara cucharilla cucharita cucharitas pizcas pizca pellizco chorros chorro chorrito punados punado dientes
diente latas lata botes bote botellas botella paquetes paquete sobres sobre manojos manojo ramas rama ramitas ramita
hojas hoja rebanadas rebanada rodajas rodaja lonchas loncha tiras tira trozos trozo filetes filete barras barra cabezas
cabeza piezas pieza unidades unidad porciones porcion cubos cubo terrones terron gotas gota tabletas tableta bolsas bolsa
frascos frasco cajas caja tallos tallo ramilletes ramillete rollos rollo bulbos bulbo medidas medida cubiletes cubilete
""".split()
# Palabras que describen la unidad ("cucharada SOPERA", "taza GRANDE").
SIZES = """sopera soperas postre rasa rasas colmada colmadas pequeno pequena pequenos pequenas grande grandes mediano mediana
medianos medianas chico chica chicos chicas cafe""".split()
# Palabras que, si quedan AL FINAL, describen el corte o el tamaño, no el ingrediente.
# OJO: aquí NO van "molido" ni "rallado". "pan rallado" y "carne molida" son ingredientes
# distintos de "pan" y "carne", y quitarlos sería perder información de verdad.
TRAILING = {"picado", "picada", "picados", "picadas", "cortado", "cortada", "cortados", "cortadas",
            "troceado", "troceada", "lavado", "lavada", "pelado", "pelada", "pelados", "peladas",
            "rebanado", "rebanada", "fileteado", "desmenuzado", "deshebrado", "limpio", "limpia",
            "extra", "virgen", "grande", "grandes", "mediano", "mediana", "medianos", "pequeno", "pequena",
            "chico", "chica", "en", "de", "y", "o", "u", "con", "sin", "al", "a", "del", "para", "que", "su",
            "lo", "le", "si", "es", "como", "cuando", "mientras", "otro", "otra"}
# Palabras de enlace que a veces quedan AL PRINCIPIO porque el renglón venía partido
# ("...,2 cucharadas con jugo de limones" -> "con jugo de limon").
LEADING_LINK = {"de", "del", "con", "sin", "y", "o", "u", "e", "al", "a", "en", "que", "como",
                "cuando", "por", "para", "lo", "la", "le", "el", "los", "las", "un", "una"}

# ---- Cantidades "vagas": ocupan espacio y no dicen nada del ingrediente -------------
# Antes se quedaban pegadas y teníamos 331 recetas cuyo ingrediente era, literalmente, "al gusto sal".
VAGUE_AMOUNT = re.compile(r"""(?ix)
    \b(?:
        al\s+gusto | a\s+gusto | a\s+su\s+gusto
      | cantidad\s+(?:necesaria|suficiente|al\s+gusto|deseada)
      | (?:el|la|lo|los|las)\s+(?:que\s+)?(?:sea\s+)?necesari[oa]s?
      | (?:el|la)\s+que\s+(?:tengas?|quieras?|guste)
      | por\s+(?:persona|comensal|cabeza) | personas? | comensal(?:es)?
      | si\s+es\s+necesari[oa] | necesari[oa]s? | cantidad(?:es)?
      | suficiente | opcional\w* | aprox(?:imadamente)? | c\.?\s?n\.?
      | recien | finamente | previamente | preferiblemente
    )\b
""")

# ---- "para ..." solo se corta cuando es un PROPÓSITO ---------------------------------
# Fíjate que "hornear" NO está en la lista, y es a propósito: la versión anterior cortaba en
# cualquier "para" y convertía "polvo para hornear" en "polvo" en 1,032 recetas.
PURPOSE = re.compile(r"""(?ix)
    (?<!polvo)\s+para\s+(?:
        decorar | adornar | acompa[ñn]ar | servir | espolvorear | engrasar | enharinar
      | fre[ií]r | rebozar | empanizar | empanar | pintar | barnizar | montar | rellenar
      | cubrir | marinar | fundir | derretir | sazonar | aderezar | remojar | cocer | hervir
      | el\b | la\b | los\b | las\b | \d
    )
""")

# A veces el rendimiento viene pegado al primer ingrediente ("Para 4 personas 350 g de macarrones").
SERVINGS_PREFIX = re.compile(
    r"^\s*para\s+\d+\s*(?:[-–aoy/]\s*\d+)?\s*(?:personas?|comensales|raciones?|porciones?)\s*", re.I)

UNIT_RX = "|".join(sorted(map(re.escape, UNITS), key=len, reverse=True))
SIZE_RX = "|".join(sorted(map(re.escape, SIZES), key=len, reverse=True))
START_QUANTITY = re.compile(r"^[\d\s/.%\u2044\-\u2013]+")     # 2, 1/2, 1.5, 1⁄2 ("½" al quitar acentos), 2-3, 0%
START_UNIT = re.compile(rf"^(?:{UNIT_RX})\b\.?\s*(?:(?:{SIZE_RX})\b\s*)*(?:de\s+|del\s+)*")
# Artículos y conjunciones sueltas. "y"/"o" aparecen en renglones como "1 y medio de agua".
START_ARTICLE = re.compile(r"^(?:de|del|la|el|los|las|un|una|unos|unas|uno|y|o|u|e|medios?|medias?)\s+")


def tokenize(line):
    """Deja el renglón en palabras limpias: sin cantidad, sin unidad y sin frases vagas."""
    text = normalize_text(line)                                  # minúsculas y sin acentos
    text = SERVINGS_PREFIX.sub("", text)                         # "Para 4 personas ..." al inicio
    text = re.sub(r"(?<=\d),(?=\d)", ".", text)                 # decimales: "1,5" -> "1.5"
    text = re.sub(r"\(.*?\)|\[.*?\]", " ", text)               # fuera lo que va entre paréntesis
    text = VAGUE_AMOUNT.sub(" ", text)                           # fuera "al gusto", "cantidad necesaria"...
    text = re.split(r"[,;]", text)[0]                            # lo que va antes de la primera coma
    text = PURPOSE.split(text)[0]                                # "... para decorar" ya no es el ingrediente
    text = re.split(r"\ssegun\b", text)[0]
    # Cantidad + unidad + artículo, repetido: hace falta para "1 taza 1/2 de azúcar",
    # que con una sola pasada se quedaba en "de azucar".
    for _ in range(4):
        nuevo = START_ARTICLE.sub("", START_UNIT.sub("", START_QUANTITY.sub("", text).strip()).strip()).strip()
        if nuevo == text:
            break
        text = nuevo
    text = re.sub(r"[^a-z0-9 ]", " ", text)                      # fuera símbolos
    text = re.sub(r"\b\d+\b", " ", text)                        # fuera números sueltos ("harina 0000")
    return text.split()

In [ ]:
# Paso 1: contamos todas las palabras que aparecen en los ingredientes del catálogo.
WORD_COUNTS = Counter(
    palabra
    for raw, src_name in zip(df["ingredients"], df["source"])
    for line in split_ingredient_lines(raw, src_name) if is_ingredient_line(line)
    for palabra in tokenize(line))

# Palabras que terminan en "s" pero NO son plural.
INVARIABLE = {"anis", "cuscus", "arroz", "maiz", "gas", "glas", "mas", "menos", "ingles", "frances",
              "tres", "dos", "seis", "despues", "atras", "jerez"}
_SINGULAR_CACHE = {}


def to_singular(word):
    """Pasa una palabra a singular eligiendo la opción que SÍ existe en el catálogo."""
    if word in _SINGULAR_CACHE:
        return _SINGULAR_CACHE[word]
    resultado = word
    if len(word) > 4 and word.endswith("s") and word not in INVARIABLE:
        opciones = [word[:-1]]                                  # "tomates" -> "tomate"
        if word.endswith("es"):
            opciones.append(word[:-2])                          # "frijoles" -> "frijol"
        if word.endswith("ces"):
            opciones.append(word[:-3] + "z")                    # "nueces"   -> "nuez"
        # Nos quedamos con la opción más usada en las recetas (si hay empate, la más larga).
        mejor = max(opciones, key=lambda w: (WORD_COUNTS.get(w, 0), len(w)))
        if WORD_COUNTS.get(mejor, 0) > 0:
            resultado = mejor
        elif word.endswith("es") and re.search(r"[lnrdjzsxy]$", word[:-2]) and len(word) > 5:
            resultado = word[:-2]                               # regla de respaldo para "-es"
        elif len(word[:-1]) >= 4:
            resultado = word[:-1]
    _SINGULAR_CACHE[word] = resultado
    return resultado

In [ ]:
# ---- El ingrediente base ------------------------------------------------------------
SYNONYMS = {
    # Formas distintas de escribir lo mismo. Se unen para que el modelo las vea como un ingrediente.
    "aceite de oliva virgen": "aceite de oliva", "aceite de oliva extra": "aceite de oliva",
    "aceite oliva virgen": "aceite de oliva", "aceite virgen": "aceite de oliva",
    "aceite de oliva suave": "aceite de oliva", "aceite oliva": "aceite de oliva",
    "polvo de hornear": "polvo para hornear", "polvo hornear": "polvo para hornear",
    "levadura quimica": "polvo para hornear",
    "azucar gla": "azucar glas", "azucar glace": "azucar glas", "azucar glase": "azucar glas",
    "azucar impalpable": "azucar glas", "azucar pulverizada": "azucar glas", 
    "gambas": "camarón", "gamba":"camarón",
}


def core_ingredient(line):
    """'2 tazas de Harina de trigo, cernida' -> 'harina de trigo'."""
    words = [to_singular(w) for w in tokenize(line)]
    while words and words[0] in LEADING_LINK:               # fuera los enlaces del principio
        words.pop(0)
    words = words[:4]                                       # máximo 4 palabras
    while words and words[-1] in TRAILING:                  # fuera las palabras de corte/tamaño del final
        words.pop()
    name = " ".join(words)
    return SYNONYMS.get(name, name)

In [ ]:
# Paso 1: separar los renglones de cada receta (una sola vez, se usa varias veces abajo).
eda["ing_lines"] = [split_ingredient_lines(raw, src) for raw, src in zip(eda["ingredients"], eda["source"])]

# Paso 2: primera pasada de nombres base, SIN partir todavía los renglones dobles.
#         Sirve para saber qué nombres existen de verdad en el catálogo.
primera_pasada = [[core_ingredient(line) for line in lines if is_ingredient_line(line)]
                  for lines in eda["ing_lines"]]
HEAD_COUNTS = Counter(nombre for lista in primera_pasada for nombre in set(lista))


def split_pair(name):
    """'sal y pimienta' -> ['sal', 'pimienta'], pero solo si las dos mitades son ingredientes de verdad.

    Hay 1,412 recetas cuyo renglón dice "sal y pimienta". Si lo dejamos entero, la búsqueda
    "tengo sal" no lo encontraría. Partimos solo cuando las dos mitades aparecen por su cuenta
    en al menos 100 recetas: así "sal y pimienta" sí se parte, pero "pimiento rojo y verde" no
    (porque "verde" solo no es un ingrediente).
    """
    if " y " not in name or len(name.split()) > 5:
        return [name]
    izquierda, _, derecha = name.partition(" y ")
    if HEAD_COUNTS.get(izquierda, 0) >= 100 and HEAD_COUNTS.get(derecha, 0) >= 100:
        return [izquierda, derecha]
    return [name]


def core_list(lines):
    """Lista final de ingredientes base de una receta: sin títulos, sin vacíos y sin repetidos."""
    result = []
    for line in lines:
        if not is_ingredient_line(line):                 # "Para los frijoles:" es un título
            continue
        for name in split_pair(core_ingredient(line)):
            if len(name) < 3 or name.startswith("para ") or name in result:
                continue
            result.append(name)
    return result


eda["core_list"] = eda["ing_lines"].map(core_list)
eda["n_ing_lines"] = eda["ing_lines"].map(len)            # renglones tal cual vienen
eda["n_ing"] = eda["core_list"].map(len)                  # ingredientes distintos ya limpios



by_source = eda.groupby("source")[["n_ing_lines", "n_ing"]].mean().round(1).sort_values("n_ing")

In [ ]:
# Lista de pasos por receta (usamos la misma función que el backend).
eda["steps"] = eda["instructions"].map(split_steps)
eda["n_steps"] = eda["steps"].map(len)
# Largo promedio de un paso, en caracteres.
eda["avg_step_chars"] = eda["steps"].map(lambda steps: np.mean([len(s) for s in steps]) if steps else np.nan)

## 2. Construcción de las columnas

In [ ]:
feat = df.drop(columns=["rating"]).copy()

assert "rating" not in feat.columns

In [ ]:
MIN_INGREDIENTS = 2

feat["core_list"] = eda["core_list"]                       # ingredientes base de cada receta (se calcularon en el EDA)
too_few = feat["core_list"].map(len) < MIN_INGREDIENTS


rows_before = len(feat)
feat = feat.loc[~too_few].copy()                           # nos quedamos solo con las demás

In [ ]:
feat["ingredients_core"] = feat["core_list"].map(" | ".join)
feat["n_ingredients"] = feat["core_list"].map(len)

In [ ]:
feat["n_steps"] = eda["n_steps"]          # calculados arriba (misma función que usa la app)

assert feat["n_steps"].min() >= 1, "Hay recetas sin pasos"

In [ ]:
feat["difficulty_level"] = feat["difficulty"].map(DIFFICULTY_LEVEL)                 # 1, 2, 3 o vacío
feat["difficulty_known"] = feat["difficulty_level"].notna().astype(int)             # 1 = dato real, 0 = faltaba

In [ ]:
# Posición (percentil) de cada receta entre todas: 0 = la que menos tiene, 1 = la que más tiene.
rank_ingredients = feat["n_ingredients"].rank(pct=True)
rank_steps = feat["n_steps"].rank(pct=True)

feat["effort_score"] = ((rank_ingredients + rank_steps) / 2).round(3)

# Grupo de esfuerzo (1 = el 25 % más sencillo ... 4 = el 25 % más trabajoso). Se usa para estimar el tiempo en 4.7.
feat["effort_group"] = pd.qcut(feat["effort_score"], 4, labels=[1, 2, 3, 4]).astype(int)

# ¿El effort_score se parece a lo que dice la fuente? (Spearman)
check = feat[["effort_score", "n_ingredients", "n_steps", "difficulty_level", "total_time_min"]].corr(method="spearman")

# Promedio de esfuerzo por dificultad: debe subir de fácil a reto.

In [ ]:
# Llave para agrupar recetas parecidas: dificultad (0 = desconocida) + grupo de esfuerzo.
feat["difficulty_group"] = feat["difficulty_level"].fillna(0).astype(int)
GROUP_KEYS = ["difficulty_group", "effort_group"]

In [ ]:
# ---- Estimación real: usamos TODOS los tiempos conocidos para calcular las medianas ----
group_median_all = feat.groupby(GROUP_KEYS)["total_time_min"].transform("median")       # mediana de su grupo (vacía si ninguno tiene tiempo)
global_median = feat["total_time_min"].median()
estimated = group_median_all.fillna(global_median)
estimated = (estimated / 5).round() * 5                                                 # tiempos "redondos", como los de las fuentes

feat["time_known"] = feat["total_time_min"].notna().astype(int)                         # 1 = tiempo real
feat["time_filled_min"] = feat["total_time_min"].fillna(estimated)                       # real si existe, estimado si no

# Rango de tiempo SOLO con tiempos reales.
TIME_BINS = [0, 15, 30, 60, 120, np.inf]
TIME_LABELS = ["hasta_15", "16_30", "31_60", "61_120", "mas_de_120"]
bucket = pd.cut(feat["total_time_min"], bins=TIME_BINS, labels=TIME_LABELS)
feat["time_bucket"] = bucket.astype(object).where(feat["time_known"] == 1, "desconocido")

# "Rápida" = tiempo REAL de 30 minutos o menos (misma regla que usa la app).
QUICK_MAX_MINUTES = 30
feat["is_quick"] = ((feat["time_known"] == 1) & (feat["total_time_min"] <= QUICK_MAX_MINUTES)).astype(int)


# La estimación no debe cambiar la forma de los datos: comparamos medianas.

In [ ]:
MIN_COUNTRY_RECIPES = 200
counts = feat["country"].value_counts()
big_countries = counts[counts >= MIN_COUNTRY_RECIPES].index

feat["country_group"] = np.where(feat["country"].isna(), "Desconocido",
                          np.where(feat["country"].isin(big_countries), feat["country"], "Otro"))

In [ ]:
# Nombre de columna: "Vegetariana" -> "tag_vegetariana"
TAG_COLUMNS = {tag: "tag_" + normalize_text(tag) for tag in TAGS}

feat_tag_lists = feat["tags"].fillna("").str.split(" | ", regex=False)
for tag, column in TAG_COLUMNS.items():
    feat[column] = feat_tag_lists.map(lambda tags, t=tag: int(t in tags))

feat["n_tags"] = feat[list(TAG_COLUMNS.values())].sum(axis=1)
feat["has_meal_tag"] = feat[[TAG_COLUMNS[t] for t in MEAL_TAGS]].max(axis=1)

In [ ]:
# --- Picante: palabras de chiles y salsas picantes (se buscan como palabras completas) ---
SPICY_WORDS = """chile chiles jalapeno jalapenos habanero habaneros serrano serranos chipotle chipotles guajillo guajillos pasilla
ancho arbol piquin cayena guindilla guindillas rocoto picante picantes harissa""".split()
spicy_regex = re.compile(r"\b(?:" + "|".join(SPICY_WORDS) + r")\b")

spicy_text = (feat["name"] + " | " + feat["ingredients_core"]).map(normalize_text)
feat["is_spicy"] = spicy_text.map(lambda t: int(bool(spicy_regex.search(t))))

# --- Técnicas de cocina: se buscan en los pasos (sin acentos y en minúsculas) ---
steps_text = feat["instructions"].map(normalize_text)
OVEN = re.compile(r"\bhorn(?:o|ea\w*|ee\w*)\b")                                   # horno, hornear, hornea, hornee...
FRYING = re.compile(r"\b(?:freir|frie|fria|frias|frien|frito|frita|fritos|fritas|freidora)\b")
BLENDER = re.compile(r"\b(?:licu\w+|procesador|procesa\w*|tritura\w*|batidora de vaso|minipimer)\b")

feat["needs_oven"] = steps_text.map(lambda t: int(bool(OVEN.search(t))))
feat["needs_frying"] = steps_text.map(lambda t: int(bool(FRYING.search(t))))
feat["needs_blender"] = steps_text.map(lambda t: int(bool(BLENDER.search(t))))

flags = ["is_spicy", "needs_oven", "needs_frying", "needs_blender"]

# Revisión a ojo: ¿tienen sentido los ejemplos?

In [ ]:
def clean_for_text(text):
    """Minúsculas, sin acentos y sin símbolos; deja solo letras, números y espacios."""
    return re.sub(r"[^a-z0-9 ]+", " ", normalize_text(text))


# Palabras de propaganda que aparecen en los títulos y no describen el plato.
# Las quitamos del nombre para que no "emparejen" recetas que no tienen nada que ver.
TITLE_FLUFF = re.compile(
    r"\b(?:paso a paso|recetas? de|recetas?|facil(?:isim[oa])?|faciles|rapid[oa]s?|"
    r"caser[oa]s?|delicios[oa]s?|riquisim[oa]s?|sencill[oa]s?|exquisit[oa]s?|"
    r"super|la mejor|el mejor|los mejores|las mejores|en casa|espectacular|increible|"
    r"perfect[oa]s?|ideal(?:es)?|especial(?:es)?)\b", re.I)

name_part = feat["name"].map(clean_for_text).str.replace(TITLE_FLUFF, " ", regex=True)
ingredients_part = feat["core_list"].map(lambda names: clean_for_text(" ".join(names)))
tags_part = feat["tags"].fillna("").map(clean_for_text)
# "Internacional" no es un país: es "no sabemos de dónde es". Se queda fuera del texto.
country_part = feat["country"].fillna("").replace("Internacional", "").map(clean_for_text)

# Unimos todo con espacios y limpiamos espacios dobles.
feat["recipe_text"] = (name_part + " " + ingredients_part + " " + tags_part + " " + country_part
                       ).str.replace(r"\s+", " ", regex=True).str.strip()

In [ ]:
def make_recipe_key(source, url, name):
    """Código de 12 caracteres que depende solo del contenido (misma receta = mismo código)."""
    raw = f"{source}|{url}|{normalize_text(name)}"
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()[:12]


feat["recipe_key"] = [make_recipe_key(s, u, n) for s, u, n in zip(feat["source"], feat["url"], feat["name"])]

assert feat["recipe_key"].is_unique, "Hay recetas distintas con el mismo recipe_key"

## 3. Dataset final, validación y guardado

In [ ]:
# Columnas finales, en el orden en que se guardan.
IDENTIFIERS = ["recipe_id", "recipe_key", "name", "source", "url"]
SIZE_AND_EFFORT = ["n_ingredients", "n_steps", "effort_score"]
DIFFICULTY_COLS = ["difficulty", "difficulty_level", "difficulty_known"]
TIME_COLS = ["total_time_min", "time_known", "time_filled_min", "time_bucket", "is_quick"]
COUNTRY_COLS = ["country", "country_group"]
TAG_COLS = ["tags"] + list(TAG_COLUMNS.values()) + ["n_tags", "has_meal_tag"]
FLAG_COLS = ["is_spicy", "needs_oven", "needs_frying", "needs_blender"]
TEXT_COLS = ["ingredients_core", "recipe_text"]

FINAL_COLUMNS = (IDENTIFIERS + SIZE_AND_EFFORT + DIFFICULTY_COLS + TIME_COLS + COUNTRY_COLS
                 + TAG_COLS + FLAG_COLS + TEXT_COLS)

model_df = feat[FINAL_COLUMNS].reset_index(drop=True).copy()      # recipe_id se conserva (no se renumera)

# Tipos pequeños y claros: los 0/1 como enteros chicos, el nivel de dificultad con vacíos permitidos.
binary_cols = [c for c in model_df.columns if c.startswith("tag_")] + ["difficulty_known", "time_known", "is_quick", "has_meal_tag"] + FLAG_COLS
model_df[binary_cols] = model_df[binary_cols].astype("int8")
model_df["n_tags"] = model_df["n_tags"].astype("int8")
model_df["difficulty_level"] = model_df["difficulty_level"].astype("Int8")

In [ ]:
# --- Identificadores ---
assert model_df["recipe_id"].is_unique and model_df["recipe_key"].is_unique
assert model_df["recipe_id"].isin(df["recipe_id"]).all(), "Hay recetas que no existen en recetas_limpias.csv"

# --- Nada vacío donde no debe ---
for column in ["name", "source", "url", "ingredients_core", "recipe_text", "n_ingredients", "n_steps", "time_filled_min", "effort_score"]:
    assert model_df[column].notna().all(), f"Nulos en {column}"
assert (model_df["n_ingredients"] >= MIN_INGREDIENTS).all() and (model_df["n_steps"] >= 1).all()
assert (model_df["time_filled_min"] > 0).all() and (model_df["time_filled_min"] <= 720).all()
assert model_df["effort_score"].between(0, 1).all()

# --- Columnas que no deben existir ---
assert "rating" not in model_df.columns
assert "avg_step_chars" not in model_df.columns
assert not model_df["recipe_text"].str.contains("Ã|Â").any(), "Texto con caracteres rotos"

# --- Los ingredientes base no deben traer basura ---
BASURA = re.compile(
    r"(?:^|\| )\s*(?:al gusto|a gusto|cantidad|suficiente|opcional|persona|recien|"
    r"para(?: |$)|polvo(?: \||$)|de |y |o |con |sin )"
    r"|\b(?:necesari[oa]|nuece|frijole|calamare|ani|gla)\b", re.I)
sucios = model_df.loc[model_df["ingredients_core"].str.contains(BASURA), ["name", "ingredients_core"]]
assert sucios.empty, f"{len(sucios)} recetas con ingredientes sucios:\n{sucios.head(8)}"

# El número de ingredientes tiene que cuadrar con la lista.
assert (model_df["ingredients_core"].str.split(" | ", regex=False).map(len)
        == model_df["n_ingredients"]).all(), "n_ingredients no cuadra con ingredients_core"
# Ningún ingrediente vacío ni de una sola letra.
piezas = model_df["ingredients_core"].str.split(" | ", regex=False).explode()
assert piezas.str.len().min() >= 3, "hay ingredientes demasiado cortos"

# --- Las columnas 0/1 deben decir lo mismo que el texto de tags ---
rebuilt = model_df[list(TAG_COLUMNS.values())].apply(
    lambda row: " | ".join(tag for tag, col in TAG_COLUMNS.items() if row[col] == 1), axis=1)
original = model_df["tags"].fillna("").map(lambda s: " | ".join(t for t in TAGS if t in s.split(" | ")))
assert (rebuilt == original).all(), "tag_* no coincide con tags"

# --- Contradicciones ---
meat_cols = [TAG_COLUMNS[t] for t in PROTEIN_TAGS]
assert not ((model_df[TAG_COLUMNS["Vegetariana"]] == 1) & (model_df[meat_cols].sum(axis=1) > 0)).any(), "Vegetariana con carne"
assert not ((model_df[TAG_COLUMNS["Vegana"]] == 1) & (model_df[TAG_COLUMNS["Vegetariana"]] == 0)).any(), "Vegana sin ser Vegetariana"
assert (model_df.loc[model_df["is_quick"] == 1, "time_known"] == 1).all(), "Rápida sin tiempo real"
assert (model_df.loc[model_df["time_known"] == 1, "time_filled_min"] == model_df.loc[model_df["time_known"] == 1, "total_time_min"]).all()

In [ ]:
model_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

# Comprobación: se lee de vuelta con las mismas columnas y los mismos datos clave.
check = pd.read_csv(OUTPUT_PATH)
assert check.columns.tolist() == FINAL_COLUMNS
assert check.shape == model_df.shape
assert check["recipe_key"].is_unique and check["recipe_id"].equals(model_df["recipe_id"])